# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [1]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cpu / device: cpu


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [5]:
# 답안
import torch

# 1. 0D Tensor (스칼라): 단순 상숫값을 표현할 때 사용
t1 = torch.tensor(3.14)

# 2. 1D Tensor (벡터): 1D 데이터나 연속된 수열을 다룰 때 사용
t2 = torch.arange(1, 6)

# 3. 2D Tensor (행렬): 2차원 가중치 행렬이나 2D 데이터를 표현할 때 사용
t3 = torch.zeros((3, 4), dtype=torch.float32)

# 4. 3D Tensor: 시계열 데이터나 텍스트 데이터(batch, seq, feature)를 다룰 때 사용
t4 = torch.randn((2, 5, 10))

# 5. 4D Tensor: 이미지 배치 데이터(batch, channel, height, width)를 다룰 때 사용
t5 = torch.ones((8, 3, 32, 32), dtype=torch.uint8)

# 리스트 생성
tensors = [
    ("t1", t1),
    ("t2", t2),
    ("t3", t3),
    ("t4", t4),
    ("t5", t5)
]

# 표 형태 출력
print(f"| {'이름':<6} | {'shape':<18} | {'dim':<4} | {'dtype':<12} | {'device':<8} |")
print("-" * 60)

for name, t in tensors:
    shape_str = str(list(t.shape))
    dtype_str = str(t.dtype).replace("torch.", "")
    device_str = str(t.device)
    print(f"| {name:<6} | {shape_str:<18} | {t.dim():<4} | {dtype_str:<12} | {device_str:<8} |")

| 이름     | shape              | dim  | dtype        | device   |
------------------------------------------------------------
| t1     | []                 | 0    | float32      | cpu      |
| t2     | [5]                | 1    | int64        | cpu      |
| t3     | [3, 4]             | 2    | float32      | cpu      |
| t4     | [2, 5, 10]         | 3    | float32      | cpu      |
| t5     | [8, 3, 32, 32]     | 4    | uint8        | cpu      |


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [3]:
# 답안
import torch

# (3, 4)와 (4, 2) Tensor 생성
tensor_a = torch.randn(3, 4)
tensor_b = torch.randn(4, 2)

# 1. 행렬곱 수행
matmul_result = torch.matmul(tensor_a, tensor_b)

# 2. 첫 번째 Tensor(tensor_a)의 연산
# 열별 평균: 행방향(dim=0)으로 축소하여 계산
col_mean = tensor_a.mean(dim=0)

# 행별 합계: 열방향(dim=1)으로 축소하여 계산
row_sum = tensor_a.sum(dim=1)

# 결과 출력
print("=== 생성된 Tensor ===")
print("Tensor A (3, 4):\n", tensor_a)
print("\nTensor B (4, 2):\n", tensor_b)

print("\n=== 연산 결과 ===")
print("1. 행렬곱 결과:\n", matmul_result)
print("   행렬곱 결과 shape:", matmul_result.shape)

print("\n2. 첫 번째 Tensor 열별 평균 (dim=0):", col_mean)
print("   첫 번째 Tensor 행별 합계 (dim=1):", row_sum)

=== 생성된 Tensor ===
Tensor A (3, 4):
 tensor([[-0.2648, -0.3737,  1.6180, -0.7262],
        [-0.2256, -0.4991,  0.3023,  0.2690],
        [-0.9008,  0.3132,  0.1176,  1.6681]])

Tensor B (4, 2):
 tensor([[-0.4092, -0.7910],
        [-0.7587,  0.0395],
        [ 1.1868, -1.3417],
        [-0.2730, -0.5982]])

=== 연산 결과 ===
1. 행렬곱 결과:
 tensor([[ 2.5104, -1.5417],
        [ 0.7563, -0.4077],
        [-0.1848, -0.4307]])
   행렬곱 결과 shape: torch.Size([3, 2])

2. 첫 번째 Tensor 열별 평균 (dim=0): tensor([-0.4637, -0.1865,  0.6793,  0.4037])
   첫 번째 Tensor 행별 합계 (dim=1): tensor([ 0.2533, -0.1534,  1.1982])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [4]:
# 답안
import torch
import time

# 크기가 큰 행렬 생성 (5000 x 5000)
size = (5000, 5000)

# 1. CPU 연산 시간 측정
a_cpu = torch.randn(size, device='cpu')
b_cpu = torch.randn(size, device='cpu')

start_time = time.time()
result_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = time.time() - start_time

print(f"CPU 연산 시간: {cpu_time:.4f} 초")

# 2. GPU 연산 시간 측정 (GPU 사용 가능 여부 확인)
if torch.cuda.is_available():
    device = torch.device('cuda')

    # GPU 메모리로 이동
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    # Warm-up (GPU 초기화 실행)
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start_time = time.time()
    result_gpu = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()  # GPU 연산 완료 대기
    gpu_time = time.time() - start_time

    print(f"GPU 연산 시간: {gpu_time:.4f} 초")
    print(f"속도 차이: GPU가 CPU보다 약 {cpu_time / gpu_time:.1f}배 빠름")

    # 차이 설명
    print("\n[설명]")
    print("CPU는 단일 연산 속도가 빠른 적은 수의 코어로 순차 처리에 최적화되어 있지만, ")
    print("GPU는 수천 개의 코어로 구성되어 대규모 데이터의 병렬 행렬 연산을 훨씬 빠르게 처리합니다.")

else:
    print("\n[GPU 사용 불가능 - CPU 전용 결과]")
    print(f"CPU 실행 결과만 측정됨: {cpu_time:.4f} 초")
    print("이유: Colab 런타임 유형이 'CPU'로 설정되어 있어 CUDA Device를 찾을 수 없습니다.")
    print("Colab 상단 메뉴의 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하면 GPU 연산을 테스트할 수 있습니다.")

CPU 연산 시간: 3.8810 초

[GPU 사용 불가능 - CPU 전용 결과]
CPU 실행 결과만 측정됨: 3.8810 초
이유: Colab 런타임 유형이 'CPU'로 설정되어 있어 CUDA Device를 찾을 수 없습니다.
Colab 상단 메뉴의 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하면 GPU 연산을 테스트할 수 있습니다.


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [6]:
# 답안
import torch

print("--- 1. y = 2x^3 + 5x (x = 3 일 때) ---")
x1 = torch.tensor(3.0, requires_grad=True)
y1 = 2 * (x1 ** 3) + 5 * x1

# 역전파 계산
y1.backward()

autograd_1 = x1.grad.item()
# 손계산: dy/dx = 6x^2 + 5 -> x=3 일 때 6*(3^2) + 5 = 59
hand_1 = 6 * (3.0 ** 2) + 5

print(f"Autograd 결과 : {autograd_1}")
print(f"손계산 결과   : {hand_1}")
print(f"일치 여부     : {autograd_1 == hand_1}")


print("\n--- 2. 임의 수식: y = 3x^2 - 4x + 1 (x = 2 일 때) ---")
x2 = torch.tensor(2.0, requires_grad=True)
y2 = 3 * (x2 ** 2) - 4 * x2 + 1

# 역전파 계산
y2.backward()

autograd_2 = x2.grad.item()
# 손계산: dy/dx = 6x - 4 -> x=2 일 때 6*(2) - 4 = 8
hand_2 = 6 * 2.0 - 4

print(f"Autograd 결과 : {autograd_2}")
print(f"손계산 결과   : {hand_2}")
print(f"일치 여부     : {autograd_2 == hand_2}")

--- 1. y = 2x^3 + 5x (x = 3 일 때) ---
Autograd 결과 : 59.0
손계산 결과   : 59.0
일치 여부     : True

--- 2. 임의 수식: y = 3x^2 - 4x + 1 (x = 2 일 때) ---
Autograd 결과 : 8.0
손계산 결과   : 8.0
일치 여부     : True
